# Music Studio Dataset — automatic private save

Run this notebook **once**, with **Accelerator None** and **Internet ON**.
Enable your private Kaggle Secret **KAGGLE_API_TOKEN** for this notebook.
The cell checks authentication first, builds the Linux x86-64/T4 runtime,
and automatically creates **one private Music Studio Dataset** containing
the runtime and all selected tools/models (about 24.77 GB of models plus runtime).
It uploads one verified file at a time; the complete model set is never held
twice on local disk. The first compilation takes time. Progress and errors
remain under `/kaggle/working/music_studio_dataset` and the named build log.

When **PRIVATE MUSIC STUDIO DATASET SUBMITTED** appears, wait for Kaggle processing.
Then import **Music-Studio.ipynb** and attach that ONE Dataset via **Add Input →
Your Datasets**. No manual Dataset upload or PC model download is required.
This is a separate GGUF dataset; your existing FP16 YuE2 assets remain usable
with their original notebooks. This builder generates no music and uses no GPU.

Set `DATASET_OWNER` only if your private credentials do not reveal your username.
The uploader follows the existing Image Studio direct private-create method.
An interrupted final request is not proof of creation; check the Dataset page
before retrying. Existing private Datasets are never overwritten.

[Music Studio guide PDF](https://github.com/bothashaun-maker/Music/blob/main/Music-Studio-Guide.pdf)
· [Setup and validation](https://github.com/bothashaun-maker/Music/blob/main/docs/music-studio/README.md)


In [ ]:
DATASET_OWNER = ''  # Your Kaggle username only if auto-detection needs it.
BUILD_JOBS = 2  # CPU compilation; lower to 1 if the build reports insufficient RAM.

# Self-contained support; no repository clone or separate repair cell required.
import base64, hashlib, json, os, signal, subprocess, sys, zlib
from pathlib import Path
packed = base64.b64decode('...')
assert hashlib.sha256(packed).hexdigest() == '4f6a6c2a5cbe53547791138732d3774398641a41375358abb1ddbb322802d068', 'Embedded Music Studio source checksum failed'
support = Path('/kaggle/working/.music_studio_support/4f6a6c2a5cbe')
for relative, contents in json.loads(zlib.decompress(packed)).items():
    path = support / relative
    if not path.resolve().is_relative_to(support.resolve()):
        raise RuntimeError('Invalid embedded source path')
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents)
task_env = dict(os.environ)
task_env['PYTHONPATH'] = str(support) + os.pathsep + task_env.get('PYTHONPATH', '')
task_env['MUSIC_STUDIO_DATASET_OWNER'] = DATASET_OWNER
task_env['MUSIC_STUDIO_BUILD_JOBS'] = str(BUILD_JOBS)

task = subprocess.Popen([sys.executable, '-u', '-m', 'music_studio.dataset'], env=task_env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, start_new_session=True)
try:
    for line in task.stdout:
        print(line, end='', flush=True)
    code = task.wait()
    if code:
        raise RuntimeError('Music Studio stopped with exit code ' + str(code) + '; inspect the error above and saved logs.')
except KeyboardInterrupt:
    print('Stopping this Music Studio process. Saved songs are retained.', flush=True)
finally:
    if task.poll() is None:
        os.killpg(task.pid, signal.SIGTERM)
        try:
            task.wait(timeout=25)
        except subprocess.TimeoutExpired:
            os.killpg(task.pid, signal.SIGKILL)
            task.wait(timeout=5)
